В этом ноутбуке мы будем делать регрессию для таргета IC50, соответсвенно от остальных целевых избавимся здесь

In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_pickle('/content/dataset_after_eda.pkl')
df.head(3)

,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,...,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,IC50_above_median,CC50_above_median,SI_above_median,SI_above_8
0,6.239374,175.482382,28.125,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,340.300,...,0,0,0,0,3,0,0,0,1,1
1,0.771831,5.402819,7.000,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,340.300,...,0,0,0,0,3,0,0,0,1,0
2,223.808778,161.142320,0.720,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,388.344,...,0,0,0,0,3,0,1,0,0,0


In [4]:
df.shape

(990, 211)

In [5]:
df = df.drop_duplicates()

In [6]:
df.shape

(958, 211)

UPD2: Я не знаю почему в eda показало что дупликатов 0, просто решил еще раз проверить. Работать в разных файлах все таки проблематично
UPD3: Все оказалось банально и просто, я сначала проверил дупликаты, затем убрал Unnamed, из - за невнимательности это проглядел и сделал себе кучу проблем, с Unnamed: 0 в наборе нет двух строк, одинаковых по всем 214 колонкам т.е. индекс всегда будет разный и соответственно и строки тоже

В Eda мы договорились логарфмировать таргеты, но это нужно только для линейных моделей. Нелинейным плевать, масштабиируемы признаки, немасштабируемы.Более того, брать будем отрицательный логарифм, тк по стандарту же делается прогноз чем больше тем лучше, ну по крайней мере часто. Но наша задача минимизировать IC50, тк он в знаменателе.Поэтому чтобы было легче просто воспринимать результаты инвертируем логарифм с помощью знака '-'.

Так же очевидно избавимся от таргетов в матрице признаков

In [7]:
y = -np.log10(df['IC50, mM'])
X = df.drop(['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8'], axis=1)

Разделим обучающие и тестовые подвыборки

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape[0])
print(X_test.shape[0])

766
192


И так, начнем

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("regressor", LinearRegression())]),
    "Ridge Regression": Pipeline([("scaler", StandardScaler()), ("regressor", Ridge())]),
    "Lasso Regression": Pipeline([("scaler", StandardScaler()), ("regressor", Lasso(alpha=0.1))]),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False) # verbose=0, чтобы не спамить логами
}

baseline_results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_log = model.predict(X_test)

    mae_log = mean_absolute_error(y_test, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
    r2 = r2_score(y_test, y_pred_log)

    y_test_real = 10 ** (-y_test)
    y_pred_real = 10 ** (-y_pred_log)

    mae_real = mean_absolute_error(y_test_real, y_pred_real)

    baseline_results.append({
        "Model": name,
        "MAE (pIC50)": round(mae_log, 4),
        "RMSE (pIC50)": round(rmse_log, 4),
        "MAE (реальный, mM)": round(mae_real, 4),
        "R2 Score": round(r2, 4)
    })

df_baseline = pd.DataFrame(baseline_results).sort_values(by="R2 Score", ascending=False)
df_baseline

,Model,MAE (pIC50),RMSE (pIC50),"MAE (реальный, mM)",R2 Score
5,CatBoost,0.4997,0.6445,193.2429,0.5220
4,Gradient Boosting,0.5098,0.6510,195.8693,0.5123
3,Random Forest,0.5181,0.6686,204.7826,0.4854
1,Ridge Regression,0.5818,0.7607,244.8503,0.3339
0,Linear Regression,0.5929,0.7803,231.9055,0.2992
2,Lasso Regression,0.6821,0.8543,239.0610,0.1600


Как и ожидалось, полный провал на линейной регрессии. Зависимость между структурой хим. соединений и биологической эффективностью сложнее чем просто линейность. Лучший результат показал случайный лес, как в принципе и все ансамбли, просто чуть хуже. Учитывая что это параметры по умолчанию результат нормальный. Так же стоит пояснить про разыменование логарифма в метрике MAE real mM, хотелось посмотреть, на не сглаженные ошибки.

In [10]:
grid = {
    'n_estimators': [300, 400],
    'max_depth': [12, 15],
    'min_samples_split': [3, 4],
    'min_samples_leaf': [2]
}

rf_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Random Forest")
rf_search.fit(X_train, y_train)

print(rf_search.best_params_)
best_rf_model = rf_search.best_estimator_
y_pred_best_rf = best_rf_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_rf))

Random Forest
Fitting 3 folds for each of 8 candidates, totalling 24 fits
{'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 300}
R2: 0.4903780689714252


In [11]:
from sklearn.ensemble import GradientBoostingRegressor

gb_param_grid = {
    'n_estimators': [150, 300],
    'learning_rate': [0.03, 0.1],
    'max_depth': [4, 6, 8],
    'min_samples_leaf': [2]
}

gb_grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=gb_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Gradient Boosting")
gb_grid_search.fit(X_train, y_train)
print(gb_grid_search.best_params_)
best_gb_model = gb_grid_search.best_estimator_
y_pred_best_gb = best_gb_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_gb))

Gradient Boosting
Fitting 3 folds for each of 12 candidates, totalling 36 fits
{'learning_rate': 0.03, 'max_depth': 4, 'min_samples_leaf': 2, 'n_estimators': 150}
R2: 0.5274259766893621


In [12]:
from catboost import CatBoostRegressor

cat_param_grid = {
    'iterations': [300, 600],
    'learning_rate': [0.03, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [3]
}

cat_grid_search = GridSearchCV(
    estimator=CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False), # отключаем спам в консоль и создание папок
    param_grid=cat_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("CatBoost")
cat_grid_search.fit(X_train, y_train)
print(cat_grid_search.best_params_)
best_cat_model = cat_grid_search.best_estimator_
y_pred_best_cat = best_cat_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_cat))

CatBoost
Fitting 3 folds for each of 12 candidates, totalling 36 fits
{'depth': 6, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.03}
R2: 0.5537129036654109


In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print(rf_search.best_params_)
best_rf_model = rf_search.best_estimator_
y_pred_best_rf = best_rf_model.predict(X_test)

mae_log = mean_absolute_error(y_test, y_pred_best_rf)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_best_rf))
r2 = r2_score(y_test, y_pred_best_rf)

y_test_real = 10 ** (-y_test)
y_pred_real = 10 ** (-y_pred_best_rf)
mae_real = mean_absolute_error(y_test_real, y_pred_real)

print("R2:",r2)
print("MAE",mae_log)
print("RMSE",rmse_log)
print("MAE",mae_real)

{'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 300}
R2: 0.4903780689714252
MAE 0.5179455123167273
RMSE 0.6654293903401599
MAE 204.37885334554508


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print(gb_grid_search.best_params_)
best_gb_model = gb_grid_search.best_estimator_
y_pred_best_gb = best_gb_model.predict(X_test)

mae_log_gb = mean_absolute_error(y_test, y_pred_best_gb)
rmse_log_gb = np.sqrt(mean_squared_error(y_test, y_pred_best_gb))
r2_gb = r2_score(y_test, y_pred_best_gb)

y_test_real = 10 ** (-y_test)
y_pred_real_gb = 10 ** (-y_pred_best_gb)
mae_real_gb = mean_absolute_error(y_test_real, y_pred_real_gb)

print("R2:", r2_gb)
print("MAE", mae_log_gb)
print("RMSE", rmse_log_gb)
print("MAE real", mae_real_gb)

{'learning_rate': 0.03, 'max_depth': 4, 'min_samples_leaf': 2, 'n_estimators': 150}
R2: 0.5274259766893621
MAE 0.4995053942726022
RMSE 0.6407857523633036
MAE real 198.35065001372388


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print(cat_grid_search.best_params_)
best_cat_model = cat_grid_search.best_estimator_
y_pred_best_cat = best_cat_model.predict(X_test)

mae_log_cat = mean_absolute_error(y_test, y_pred_best_cat)
rmse_log_cat = np.sqrt(mean_squared_error(y_test, y_pred_best_cat))
r2_cat = r2_score(y_test, y_pred_best_cat)

y_test_real = 10 ** (-y_test)
y_pred_real_cat = 10 ** (-y_pred_best_cat)
mae_real_cat = mean_absolute_error(y_test_real, y_pred_real_cat)

print("R2:", r2_cat)
print("MAE", mae_log_cat)
print("RMSE", rmse_log_cat)
print("MAE real", mae_real_cat)

{'depth': 6, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.03}
R2: 0.5537129036654109
MAE 0.48999001255683067
RMSE 0.6227089233247337
MAE real 196.03748955529667


Random Forest:<br>
R2: 0.4903780689714252<br>
MAE 0.5179455123167273<br>
RMSE 0.6654293903401599<br>
MAE 204.37885334554508<br>

Gradient Boosting:<br>
R2: 0.5274259766893621<br>
MAE 0.4995053942726022<br>
RMSE 0.6407857523633036<br>
MAE real 198.35065001372388<br>

CatBoost:<br>
R2: 0.5537129036654109<br>
MAE 0.48999001255683067<br>
RMSE 0.6227089233247337<br>
MAE real 196.03748955529667<br>

#Вывод

Примечание:
*Долгое время я боролся с ничем. Я не понимаю почему в eda показало, что дублей нет. Изначально результаты метрик были просто ужасными, я бы показал, но заново писать сил нет. В кратце, на примере линейной регрессии, она улетала в глобальный минус, кэф детерминации был около -43372, конечно понятно что линейная регрессия не лучший выбор, НО ЧТОБЫ НАСТОЛЬКО. Такая же ситуация была и с другими метриками, но мягче. НЕлинейных эта ситуация очевидно коснулась меньше, однако результаты все равно оставляли желать лучшего.*

В качестве финальной модели утвержден CatBoost с параметрами {'depth': 6, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.03}, показавший наилучшую обобщающую способность на данных. Он показал наилучшую обобщающую способность с результатом R2 = 0.554, MAE = 0.49 и RMSE = 0.623.

После бейзлайнов и последующего точечного подбора параметров через GridSearchCV с прямой оптимизацией метрики R2 было протестировано несколько архитектур, ансамблевые методы бустинга оказались лучшими.
UPD:
Чтобы еще больше улучшить резултьтаты можно более сложный подбор гиперпараметров.

В следующих ноутбуках будем делать очистику у дублей каждый раз.